# Panda MDH Forward Kinematics Demo

This notebook demonstrates:
1. **MDH Parameters** — Modified DH (Craig convention) parameter table for Panda
2. **URDF Visualization** — Load the URDF and visualize the default configuration $q_{init} = [0, 0, 0, -0.0698, 0, 0, 0]$
3. **Animation** — Animate joint-by-joint motion from default config, verify URDF and MDH match

> Step-by-step MDH matrix derivation for each joint: see [panda_mdh_derivation.md](panda_mdh_derivation.md)

In [ ]:
import numpy as np
from numpy import pi, cos, sin
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display
import roboticstoolbox as rtb
from spatialmath import SE3

# Chinese font setup
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=6, suppress=True)

%matplotlib inline

---
## 1. Panda MDH Parameter Table

### Modified DH (Craig Convention)

The transformation from frame $i{-}1$ to frame $i$ is:

$${}^{i-1}T_i = \text{Rot}_x(\alpha_{i-1}) \cdot \text{Trans}_x(a_{i-1}) \cdot \text{Rot}_z(\theta_i) \cdot \text{Trans}_z(d_i)$$

### Parameter Table

| Joint | $a_{i-1}$ (m) | $\alpha_{i-1}$ (rad) | $d_i$ (m) | $\theta_i$ (rad) | Joint Limits (rad) |
|:-----:|:--------------:|:--------------------:|:---------:|:-----------------:|:------------------:|
| 1 | 0 | 0 | 0.333 | $\theta_1$ (init: 0) | $[-2.90,\; +2.90]$ |
| 2 | 0 | $-\pi/2$ | 0 | $\theta_2$ (init: 0) | $[-1.76,\; +1.76]$ |
| 3 | 0 | $+\pi/2$ | 0.316 | $\theta_3$ (init: 0) | $[-2.90,\; +2.90]$ |
| 4 | 0.0825 | $+\pi/2$ | 0 | $\theta_4$ (init: -0.0698) | $[-3.07,\; -0.07]$ |
| 5 | -0.0825 | $-\pi/2$ | 0.384 | $\theta_5$ (init: 0) | $[-2.90,\; +2.90]$ |
| 6 | 0 | $+\pi/2$ | 0 | $\theta_6$ (init: 0) | $[-0.02,\; +3.75]$ |
| 7 | 0.088 | $+\pi/2$ | 0 | $\theta_7$ (init: 0) | $[-2.90,\; +2.90]$ |
| Flange | 0 | 0 | 0.107 | 0 (fixed) | — |

> **Note:** These parameters come from the [Franka Emika official documentation](https://frankaemika.github.io/docs/control_parameters.html), which uses the **Modified DH (Craig)** convention. The frame placement rule is: $z_i$ axis aligns with **joint $i$** (not joint $i{+}1$ as in Standard DH).

> **Initial values:** URDF does not define initial joint positions. The values above are `joint_state_publisher_gui` defaults: all joints default to $\theta = 0$, except **Joint 4** whose range $[-3.07, -0.07]$ does not include 0, so it is clamped to the upper limit $-0.0698$.

In [ ]:
# ============================================================
#  MDH Parameters & Forward Kinematics
# ============================================================

# (a_{i-1}, alpha_{i-1}, d_i)
mdh_params = [
    (0,       0,      0.333),   # Joint 1: theta_1 (init: 0)
    (0,      -pi/2,   0    ),   # Joint 2: theta_2 (init: 0)
    (0,       pi/2,   0.316),   # Joint 3: theta_3 (init: 0)
    (0.0825,  pi/2,   0    ),   # Joint 4: theta_4 (init: -0.0698)
    (-0.0825,-pi/2,   0.384),   # Joint 5: theta_5 (init: 0)
    (0,       pi/2,   0    ),   # Joint 6: theta_6 (init: 0)
    (0.088,   pi/2,   0    ),   # Joint 7: theta_7 (init: 0)
]
flange_d = 0.107  # Flange: fixed joint, d = 0.107 m

# Default initial joint values (URDF default config)
# Joint 4 range is [-3.07, -0.07], does not include 0, clamped to upper limit -0.0698
q_init = np.array([0, 0, 0, -0.0698, 0, 0, 0])


def mdh_transform(theta, d, a_prev, alpha_prev):
    """
    MDH single-joint transform:
    T_i = Rx(alpha_{i-1}) * Tx(a_{i-1}) * Rz(theta_i) * Tz(d_i)
    """
    ca, sa = cos(alpha_prev), sin(alpha_prev)
    ct, st = cos(theta), sin(theta)

    Rx = np.array([[1, 0,  0,  0],
                   [0, ca, -sa, 0],
                   [0, sa,  ca, 0],
                   [0, 0,   0,  1]])
    Tx = np.array([[1, 0, 0, a_prev],
                   [0, 1, 0, 0],
                   [0, 0, 1, 0],
                   [0, 0, 0, 1]])
    Rz = np.array([[ct, -st, 0, 0],
                   [st,  ct, 0, 0],
                   [0,   0,  1, 0],
                   [0,   0,  0, 1]])
    Tz = np.array([[1, 0, 0, 0],
                   [0, 1, 0, 0],
                   [0, 0, 1, d],
                   [0, 0, 0, 1]])
    return Rx @ Tx @ Rz @ Tz


def fk_mdh(q):
    """MDH forward kinematics. Returns list of frames [T_base, T_1, ..., T_7, T_flange]."""
    frames = [np.eye(4)]  # base
    T = np.eye(4)
    for i, (a, alpha, d) in enumerate(mdh_params):
        T = T @ mdh_transform(q[i], d, a, alpha)
        frames.append(T.copy())
    # flange (fixed)
    T_fl = np.eye(4); T_fl[2, 3] = flange_d
    T = T @ T_fl
    frames.append(T.copy())
    return frames


init_vals = [0, 0, 0, -0.0698, 0, 0, 0]
print('MDH parameters defined: 7 revolute joints + 1 fixed flange')
print(f'Total DOF: 7')
print(f'Default init: q = {q_init}')
print(f'\nParameter table:')
print(f'{"Joint":>7} | {"a (m)":>10} | {"alpha (rad)":>12} | {"d (m)":>8} | theta')
print('-' * 75)
for i, (a, alpha, d) in enumerate(mdh_params):
    alpha_str = f'{alpha/pi:+.1f}pi' if alpha != 0 else '0'
    print(f'{i+1:>7} | {a:>10.4f} | {alpha_str:>12} | {d:>8.3f} | theta_{i+1} (init: {init_vals[i]})')
print(f'{"Flange":>7} | {0:>10.4f} | {"0":>12} | {flange_d:>8.3f} | 0 (fixed)')

---
## 2. Load URDF & Visualize Default Configuration

Load the Panda URDF using `roboticstoolbox` and visualize the default configuration $\mathbf{q}_{\text{init}} = [0, 0, 0, -0.0698, 0, 0, 0]$.

> **Why not all zeros?** Joint 4's limit range is $[-3.07, -0.07]$, which does not include $0$. The `joint_state_publisher_gui` clamps it to the upper limit $-0.0698 \approx -4°$.

In [ ]:
# Load URDF model
urdf_path = 'panda_description/urdf/panda_arm.urdf'
panda_urdf = rtb.ERobot.URDF(urdf_path)
print(f'URDF model loaded: {panda_urdf.name}')
print(f'  Links: {panda_urdf.n + 1}')
print(f'  DOF:   {panda_urdf.n}')

In [ ]:
# ============================================================
#  Visualize default configuration: URDF vs MDH
# ============================================================

# MDH frames (9 points: base + 7 joints + flange)
frames_mdh = fk_mdh(q_init)
pos_mdh = np.array([T[:3, 3] for T in frames_mdh])

# URDF end-effector
T_urdf_end = panda_urdf.fkine(q_init).A
pos_urdf_end = T_urdf_end[:3, 3]

# Verify match
pos_err = np.linalg.norm(pos_mdh[-1] - pos_urdf_end)
print(f'Default config q_init = {q_init}')
print(f'End-effector comparison:')
print(f'  MDH  end: {pos_mdh[-1]}')
print(f'  URDF end: {pos_urdf_end}')
print(f'  Error:    {pos_err:.2e} m  {"PASS" if pos_err < 1e-6 else "FAIL"}')

# --- Plot ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# MDH arm
ax.plot(pos_mdh[:, 0], pos_mdh[:, 1], pos_mdh[:, 2],
        '-o', color='#2196F3', linewidth=3, markersize=8,
        label='MDH FK', markerfacecolor='#2196F3',
        markeredgecolor='white', markeredgewidth=1)

# URDF end-effector marker
ax.scatter(*pos_urdf_end, color='#FF5722', s=150, marker='*',
           zorder=10, label='URDF end-effector')

# Base marker
ax.scatter(*pos_mdh[0], color='#4CAF50', s=120, marker='s',
           zorder=5, label='Base')

# Joint labels
joint_names = ['Base', 'J1', 'J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'Flange']
for i, (name, pos) in enumerate(zip(joint_names, pos_mdh)):
    ax.text(pos[0]+0.02, pos[1]+0.02, pos[2]+0.02, name, fontsize=8, color='#333')

q_str = ', '.join(f'{v}' for v in q_init)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title(f'Panda Default Configuration  $q_{{init}}$ = [{q_str}]', fontsize=13)
ax.set_xlim([-0.5, 0.5])
ax.set_ylim([-0.5, 0.5])
ax.set_zlim([0, 1.2])
ax.view_init(elev=25, azim=135)
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

---
## 3. Animation: Joint-by-Joint Motion + MDH Verification

Starting from the default configuration $\mathbf{q}_{\text{init}} = [0, 0, 0, -0.0698, 0, 0, 0]$, each joint moves independently (sinusoidal trajectory). The animation shows:
- **Blue line**: MDH forward kinematics
- **Red star**: URDF end-effector position (verification)
- **Top-right info**: MDH vs URDF position error at the end-effector

In [ ]:
# ============================================================
#  Generate trajectory: each joint moves one at a time
#  Starting from default config q_init
# ============================================================
n_steps = 20       # frames per joint
n_joints = 7
targets = [pi/4, -pi/3, pi/4, -pi/2, pi/3, pi/2, -pi/4]  # peak displacement per joint

trajectory = [q_init.copy()]  # start at default config
for j in range(n_joints):
    for k in range(1, n_steps + 1):
        t = k / n_steps
        q = q_init.copy()
        q[j] = q_init[j] + targets[j] * sin(pi * t)  # oscillate around init value
        trajectory.append(q.copy())

# Pre-compute all frames
all_mdh_pos = []
all_urdf_end = []
all_errors = []

for q in trajectory:
    frames = fk_mdh(q)
    pos = np.array([T[:3, 3] for T in frames])
    all_mdh_pos.append(pos)
    
    T_urdf = panda_urdf.fkine(q).A
    urdf_end = T_urdf[:3, 3]
    all_urdf_end.append(urdf_end)
    
    err = np.linalg.norm(pos[-1] - urdf_end)
    all_errors.append(err)

max_err = max(all_errors)
print(f'Trajectory: {len(trajectory)} frames ({n_steps} per joint x {n_joints} joints + 1)')
print(f'Starting from q_init = {q_init}')
print(f'Max MDH vs URDF end-effector error: {max_err:.2e} m  {"PASS" if max_err < 1e-6 else "FAIL"}')

In [ ]:
# ============================================================
#  Create animation (single 3D view, error as text overlay)
# ============================================================
fig = plt.figure(figsize=(10, 8))
ax3d = fig.add_subplot(111, projection='3d')
ax3d.set_xlabel('X (m)')
ax3d.set_ylabel('Y (m)')
ax3d.set_zlabel('Z (m)')
ax3d.set_title('Panda Joint-by-Joint Motion (from $q_{init}$)', fontsize=13, fontweight='bold')
lim = 0.9
ax3d.set_xlim([-lim, lim])
ax3d.set_ylim([-lim, lim])
ax3d.set_zlim([0, 1.3])
ax3d.view_init(elev=25, azim=135)

# Plot elements
line_mdh, = ax3d.plot([], [], [], '-o', color='#2196F3', linewidth=3,
                       markersize=6, label='MDH FK',
                       markerfacecolor='#2196F3', markeredgecolor='white', markeredgewidth=0.5)
scat_urdf = ax3d.scatter([], [], [], color='#FF5722', s=120, marker='*',
                          zorder=10, label='URDF end')
ax3d.legend(fontsize=10, loc='upper left')

# Info text (top-left)
info_text = ax3d.text2D(0.02, 0.95, '', transform=ax3d.transAxes, fontsize=9,
                         verticalalignment='top', fontfamily='monospace',
                         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Error text (top-right)
err_text = ax3d.text2D(0.98, 0.95, '', transform=ax3d.transAxes, fontsize=9,
                        verticalalignment='top', horizontalalignment='right',
                        fontfamily='monospace',
                        bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.8))

def init():
    line_mdh.set_data_3d([], [], [])
    info_text.set_text('')
    err_text.set_text('')
    return [line_mdh, info_text, err_text]

def update(idx):
    p = all_mdh_pos[idx]
    ue = all_urdf_end[idx]
    q = trajectory[idx]
    err = all_errors[idx]
    active = min((idx - 1) // n_steps, n_joints - 1) if idx > 0 else 0

    line_mdh.set_data_3d(p[:, 0], p[:, 1], p[:, 2])
    scat_urdf._offsets3d = ([ue[0]], [ue[1]], [ue[2]])

    q_str = ', '.join(f'{v:+.2f}' for v in q)
    info_text.set_text(
        f'Frame: {idx}/{len(trajectory)-1}\n'
        f'Active: Joint {active+1}\n'
        f'q = [{q_str}]'
    )

    err_text.set_text(
        f'MDH vs URDF\n'
        f'End-eff err: {err:.2e} m\n'
        f'Max err: {max_err:.2e} m'
    )

    return [line_mdh, info_text, err_text]

anim = FuncAnimation(fig, update, frames=len(trajectory),
                     init_func=init, interval=80, blit=False)
plt.tight_layout()
plt.close(fig)

# Display as interactive JS animation
display(HTML(anim.to_jshtml()))